# Pakistan Heart Risk Assessment - RAG System

## Complete Development Notebook

This notebook implements a production-ready **Retrieval-Augmented Generation (RAG)** system for personalized heart health risk assessment using:
- **Embeddings**: BAAI/bge-base-en-v1.5 (768 dimensions)
- **Vector DB**: Pinecone (cloud-hosted semantic search)
- **Keyword Search**: BM25 (lexical matching)
- **Hybrid Fusion**: RRF (Reciprocal Rank Fusion)
- **Re-ranking**: CrossEncoder (semantic relevance scoring)
- **LLM**: Groq's Llama 3.1 8B & 3.3 70B


## Step 1: Configure API Keys

Initialize credentials for external services:
- **PINECONE_API_KEY**: Vector database for semantic search
- **HF_TOKEN**: HuggingFace token for Spaces deployment
- **GROQ_API_KEY**: LLM provider (switched from older keys)
- **PINECONE_INDEX_NAME**: Created index embeddings 768 dimensions


In [ ]:
PINECONE_API_KEY = ""
HF_TOKEN = " "
GROQ_API_KEY = ""
PINECONE_INDEX_NAME = "heart-risk-300-chunks"

## Step 2: Clean up local PDFs

Remove any PDF files that may have been left in the working directory to ensure we work with a clean state.


In [2]:
import os

# Remove ALL PDFs
for f in os.listdir():
    if f.endswith('.pdf'):
        os.remove(f)

# Verify
remaining = [f for f in os.listdir() if f.endswith('.pdf')]
print(f"PDFs remaining: {len(remaining)}")
print("All clear ✅" if len(remaining) == 0 else "Still some files remaining ❌")

PDFs remaining: 0
All clear ✅


## Step 3: Discover PDF documents

List all available PDF files in the `PDFs/` folder (medical guidelines on heart disease, hypertension, lifestyle, etc.)


In [3]:
import os

pdf_folder = r"D:\Classess\ITA\Rag_Project\Heart Attack Project\PDFs"
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]

print(f"Found PDFs: {pdf_files}")
print(f"Total: {len(pdf_files)} files")

Found PDFs: ['12919_2025_Article_350Non-communicable diseases prevention and control in Pakistan.pdf', '20+Frequency+of+ACS+among+Patients+Presenting+with+Atypical+Presentation+in+the+National+Institute+of+Cardiovascular.pdf', '2198-Article Text-8619-1-10-20220704.pdf', '3rd-Hypertension-Guideline-2018-PHL.pdf', 'arnett-et-al-2019-2019-acc-aha-guideline-on-the-primary-prevention-of-cardiovascular-disease-executive-summary-a-report.pdf', 'Biryani & Diet MVP.pdf', 'Caring For Your Heart.pdf', 'cvd-risk-non-laboratory-based-charts.pdf', 'Eating for A Healthy Heart (English)_2018.pdf', 'Emergency managemenr of acute myocardial infarction.pdf', 'main.pdf', 'pone.0185466.s003.pdf', 'south-asia.pdf', 'The Clinical Logic.pdf', 'The Risk Factor.pdf', 'Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf', 'Trans fatt acids - A risk factor for cardiovascular disease.pdf', 'ukmss-47839.pdf', 'What-Are-the-Warning-Signs-of-Heart-Attack.pdf']
Total: 19 files


## Step 4: Filter and clean PDF list

Remove duplicate files (e.g., files with "(1)" in the name) that may have been created during downloads.


In [4]:
import fitz
import os

pdf_folder = r"D:\Classess\ITA\Rag_Project\Heart Attack Project\PDFs"

# Only keep files WITHOUT (1) in the name
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf') and '(1)' not in f]
print(f"Cleaned PDF list: {len(pdf_files)} files")
print(pdf_files)

Cleaned PDF list: 19 files
['12919_2025_Article_350Non-communicable diseases prevention and control in Pakistan.pdf', '20+Frequency+of+ACS+among+Patients+Presenting+with+Atypical+Presentation+in+the+National+Institute+of+Cardiovascular.pdf', '2198-Article Text-8619-1-10-20220704.pdf', '3rd-Hypertension-Guideline-2018-PHL.pdf', 'arnett-et-al-2019-2019-acc-aha-guideline-on-the-primary-prevention-of-cardiovascular-disease-executive-summary-a-report.pdf', 'Biryani & Diet MVP.pdf', 'Caring For Your Heart.pdf', 'cvd-risk-non-laboratory-based-charts.pdf', 'Eating for A Healthy Heart (English)_2018.pdf', 'Emergency managemenr of acute myocardial infarction.pdf', 'main.pdf', 'pone.0185466.s003.pdf', 'south-asia.pdf', 'The Clinical Logic.pdf', 'The Risk Factor.pdf', 'Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf', 'Trans fatt acids - A risk factor for cardiovascular disease.pdf', 'ukmss-47839.pdf', 'What-Are-the-Warning-Signs-of-Heart-Attack.pdf']


## Step 5: Extract text from PDFs

Parse each PDF using PyMuPDF (fitz) to extract page-by-page text content. Track metadata (source filename, page number) for traceability.


In [5]:
import fitz

all_texts = []
doc_metadata = []

pdf_folder = r"D:\Classess\ITA\Rag_Project\Heart Attack Project\PDFs"

for pdf_file in pdf_files:
    try:
        full_path = os.path.join(pdf_folder, pdf_file)
        doc = fitz.open(full_path)
        num_pages = len(doc)
        for page_num in range(num_pages):
            page = doc[page_num]
            text = page.get_text()
            if text.strip():
                all_texts.append(text)
                doc_metadata.append({
                    "source": pdf_file,
                    "page": page_num + 1
                })
        doc.close()
        print(f"✅ {pdf_file} — {num_pages} pages")
    except Exception as e:
        print(f"❌ {pdf_file} — Error: {e}")

print(f"\nTotal pages extracted: {len(all_texts)}")

✅ 12919_2025_Article_350Non-communicable diseases prevention and control in Pakistan.pdf — 9 pages
✅ 20+Frequency+of+ACS+among+Patients+Presenting+with+Atypical+Presentation+in+the+National+Institute+of+Cardiovascular.pdf — 5 pages
✅ 2198-Article Text-8619-1-10-20220704.pdf — 6 pages
✅ 3rd-Hypertension-Guideline-2018-PHL.pdf — 77 pages
✅ arnett-et-al-2019-2019-acc-aha-guideline-on-the-primary-prevention-of-cardiovascular-disease-executive-summary-a-report.pdf — 33 pages
✅ Biryani & Diet MVP.pdf — 99 pages
✅ Caring For Your Heart.pdf — 7 pages
✅ cvd-risk-non-laboratory-based-charts.pdf — 22 pages
✅ Eating for A Healthy Heart (English)_2018.pdf — 6 pages
✅ Emergency managemenr of acute myocardial infarction.pdf — 15 pages
✅ main.pdf — 2 pages
✅ pone.0185466.s003.pdf — 2 pages
✅ south-asia.pdf — 2 pages
✅ The Clinical Logic.pdf — 61 pages
✅ The Risk Factor.pdf — 142 pages
✅ Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf — 5 pages
✅ Trans fatt acids - A risk factor 

## Step 6: Install text splitting library

Install `langchain-text-splitters` for advanced document chunking strategies.


In [27]:
%pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 7: Load and process Pakistani dishes nutrition data

Load CSV with nutrition facts and health recommendations for common Pakistani dishes. Format as structured text for RAG indexing.


In [6]:
import pandas as pd

csv_path = r"D:\Classess\ITA\Rag_Project\Heart Attack Project\PDFs\dishes_nutrition.csv"
dishes_df = pd.read_csv(csv_path)

for _, row in dishes_df.iterrows():
    dish_text = f"""Dish: {row['Dish']}
Nutritional Information (per 100g):
- Calories: {row['Calories_per_100g']} kcal
- Fat: {row['Fat_percent']}%
- Protein: {row['Protein_percent']}%
- Carbohydrates: {row['Carbs_percent']}%

Health Note:
- If Fat > 15%: High fat. Limit to occasional eating.
- If Fat 5-15%: Moderate fat.
- If Carbs > 20%: High carbs. Monitor if diabetic.

Source: Pakistani Dishes Nutrition Database"""
    all_texts.append(dish_text)
    doc_metadata.append({"source": "pakistani_dishes_nutrition.csv", "page": 1})

print(f"PDF pages: {len(all_texts) - len(dishes_df)}")
print(f"Dish chunks added: {len(dishes_df)}")
print(f"Total texts: {len(all_texts)}")

PDF pages: 368
Dish chunks added: 30
Total texts: 398


## Step 8: Chunk text using recursive splitting

Split documents into **300-token chunks with 50-token overlap** using a recursive character splitter. This preserves semantic boundaries while enabling focused retrieval.


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks = []
chunks_metadata = []

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

for i, text in enumerate(all_texts):
    for chunk in splitter.split_text(text):
        if chunk.strip():
            chunks.append(chunk)
            chunks_metadata.append(doc_metadata[i])

print(f"Total chunks created: {len(chunks)}")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks created: 4413


## Step 9: Install embedding library

Install `sentence-transformers` for generating dense vector embeddings.


In [30]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 10: Load embedding model

Load the **BAAI/bge-base-en-v1.5** encoder. This model produces 768-dimensional embeddings optimized for semantic similarity ranking.


In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')
print("✅ Model loaded")

# Test it
test_vector = embedding_model.encode("test sentence")
print(f"Vector dimension: {len(test_vector)}") 

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2646.29it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded
Vector dimension: 768


## Step 11: Install Pinecone SDK

Install the Pinecone Python client for vector database operations.


In [32]:
%pip install --upgrade pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from pinecone import Pinecone

# If you encounter an ImportError like 'cannot import name \'Pinecone\' from \'pinecone\'',
# it's likely due to environment inconsistencies after package installations/uninstalls.
# A common fix in Google Colab is to restart the runtime (Runtime -> Restart runtime...).
# After restarting, run all cells again.

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
print(f"✅ Connected to Pinecone index: {PINECONE_INDEX_NAME}")
print(index.describe_index_stats())

✅ Connected to Pinecone index: heart-risk-300-chunks
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '187',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 04 Apr 2026 18:06:07 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '79',
                                    'x-pinecone-request-latency-ms': '79',
                                    'x-pinecone-response-duration-ms': '81'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 4413}},
 'storageFullness': 0.0,
 'total_vector_count': 4413,
 'vector_type': 'dense'}


## Step 12: Connect to Pinecone vector database

Establish connection to pre-created Pinecone index (`heart-risk-300-chunks`) containing embeddings and metadata.


## Step 13: Embed and upload chunks to Pinecone

Generate embeddings for all chunks and upload to Pinecone in batches of 100 vectors. Each vector includes metadata (text, source, page).


In [34]:
import uuid

print("Starting embedding and upload to Pinecone...")
print(f"Total chunks to process: {len(chunks)}")

batch_size = 100  # upload 100 at a time

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    batch_meta = chunks_metadata[i:i+batch_size]

    # Generate embeddings
    vectors = embedding_model.encode(batch_chunks).tolist()

    # Prepare for Pinecone
    pinecone_vectors = []
    for j, (vec, chunk, meta) in enumerate(zip(vectors, batch_chunks, batch_meta)):
        pinecone_vectors.append({
            "id": str(uuid.uuid4()),
            "values": vec,
            "metadata": {
                "text": chunk,
                "source": meta["source"],
                "page": meta["page"]
            }
        })

    # Upload batch
    index.upsert(vectors=pinecone_vectors)
    print(f"✅ Uploaded batch {i//batch_size + 1}/{(len(chunks)//batch_size) + 1}")

print(f"\n🎉 Done! Verifying...")
stats = index.describe_index_stats()
print(f"Total vectors in Pinecone: {stats['total_vector_count']}")

Starting embedding and upload to Pinecone...
Total chunks to process: 4413
✅ Uploaded batch 1/45
✅ Uploaded batch 2/45
✅ Uploaded batch 3/45
✅ Uploaded batch 4/45
✅ Uploaded batch 5/45
✅ Uploaded batch 6/45
✅ Uploaded batch 7/45
✅ Uploaded batch 8/45
✅ Uploaded batch 9/45
✅ Uploaded batch 10/45
✅ Uploaded batch 11/45
✅ Uploaded batch 12/45
✅ Uploaded batch 13/45
✅ Uploaded batch 14/45
✅ Uploaded batch 15/45
✅ Uploaded batch 16/45
✅ Uploaded batch 17/45
✅ Uploaded batch 18/45
✅ Uploaded batch 19/45
✅ Uploaded batch 20/45
✅ Uploaded batch 21/45
✅ Uploaded batch 22/45
✅ Uploaded batch 23/45
✅ Uploaded batch 24/45
✅ Uploaded batch 25/45
✅ Uploaded batch 26/45
✅ Uploaded batch 27/45
✅ Uploaded batch 28/45
✅ Uploaded batch 29/45
✅ Uploaded batch 30/45
✅ Uploaded batch 31/45
✅ Uploaded batch 32/45
✅ Uploaded batch 33/45
✅ Uploaded batch 34/45
✅ Uploaded batch 35/45
✅ Uploaded batch 36/45
✅ Uploaded batch 37/45
✅ Uploaded batch 38/45
✅ Uploaded batch 39/45
✅ Uploaded batch 40/45
✅ Uploaded bat

## Step 14: Retrieve chunks and build BM25 index

Fetch all chunks from Pinecone and rebuild the BM25 (keyword search) index locally. BM25 will be used for keyword-based retrieval in the hybrid pipeline.


In [10]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

print("Fetching chunks from Pinecone...")

# Connect to correct index
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("heart-risk-300-chunks")

# Get total count
stats = index.describe_index_stats()
total = stats['total_vector_count']
print(f"Total vectors in Pinecone: {total}")

# Fetch all chunks using dummy vector (768 dims)
dummy_vector = [0.0] * 768

results = index.query(
    vector=dummy_vector,
    top_k=total,
    include_metadata=True
)

# Rebuild chunks and metadata
chunks = []
chunks_metadata = []

for match in results['matches']:
    chunks.append(match['metadata']['text'])
    chunks_metadata.append({
        'source': match['metadata']['source'],
        'page': match['metadata']['page']
    })

print(f"✅ Retrieved {len(chunks)} chunks from Pinecone")

# Build BM25
docs_for_bm25 = [
    Document(page_content=chunks[i], metadata=chunks_metadata[i])
    for i in range(len(chunks))
]
bm25_retriever = BM25Retriever.from_documents(docs_for_bm25, k=3)
print(f"✅ BM25 built with {len(chunks)} chunks")
print("🎉 Ready — no PDFs needed!")

Fetching chunks from Pinecone...
Total vectors in Pinecone: 4413
✅ Retrieved 4413 chunks from Pinecone
✅ BM25 built with 4413 chunks
🎉 Ready — no PDFs needed!


## Step 15: Define semantic search function

Query the Pinecone index to retrieve semantically similar documents using the embedding model.


In [11]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Semantic search function via Pinecone
def semantic_search(query, k=3):
    query_vector = embedding_model.encode(query).tolist()
    results = index.query(
        vector=query_vector,
        top_k=k,
        include_metadata=True
    )
    return [
        Document(
            page_content=match['metadata']['text'],
            metadata={
                'source': match['metadata']['source'],
                'page': match['metadata']['page'],
                'score': match['score']
            }
        )
        for match in results['matches']
    ]

print("✅ Semantic search ready")

✅ Semantic search ready


## Step 16: Define hybrid search (Reciprocal Rank Fusion)

Combine BM25 (keyword) and semantic (vector) results using **RRF (Reciprocal Rank Fusion)**:
- Score = 1/(rank + 60) from each retriever
- Sum scores for documents appearing in both results
- Re-rank and return top-k final documents

This avoids keyword-only failures and ensures semantic relevance.


In [12]:
def hybrid_search(query, k=5):

    # ===== SOURCE 1: BM25 Results =====
    bm25_results = bm25_retriever.invoke(query)
    print("BM25 RESULTS:")
    for i, doc in enumerate(bm25_results, 1):
        print(f"  {i}. {doc.metadata['source']} — {doc.page_content[:100]}...")

    # ===== SOURCE 2: Semantic Results =====
    semantic_results = semantic_search(query, k=k)
    print("\nSEMANTIC RESULTS:")
    for i, doc in enumerate(semantic_results, 1):
        print(f"  {i}. {doc.metadata['source']} — {doc.page_content[:100]}...")

    # ===== RRF COMBINATION =====
    scores = {}
    all_docs = {}

    for rank, doc in enumerate(bm25_results):
        doc_id = doc.page_content[:100]
        scores[doc_id] = scores.get(doc_id, 0) + 1/(rank + 60)
        all_docs[doc_id] = doc

    for rank, doc in enumerate(semantic_results):
        doc_id = doc.page_content[:100]
        scores[doc_id] = scores.get(doc_id, 0) + 1/(rank + 60)
        all_docs[doc_id] = doc

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    final_docs = [all_docs[doc_id] for doc_id, score in sorted_docs[:k]]

    print("\nRRF COMBINED RESULTS:")
    for i, (doc_id, score) in enumerate(sorted_docs[:k], 1):
        print(f"  {i}. Score: {score:.4f} — {all_docs[doc_id].metadata['source']} — {doc_id[:100]}...")

    return final_docs

# Test it
query = "I eat biryani and karahi 4 times a week, is that bad for my heart"
print(f"Query: {query}\n")
print("="*60)
results = hybrid_search(query)

Query: I eat biryani and karahi 4 times a week, is that bad for my heart

BM25 RESULTS:
  1. Caring For Your Heart.pdf — d. Smaller than the size of my thumb  
 
11. How is excess fat distributed across your body? 
 
a. I...
  2. What-Are-the-Warning-Signs-of-Heart-Attack.pdf — journeys with heart disease and stroke 
by joining our Support Network at 
heart.org/SupportNetwork....
  3. Caring For Your Heart.pdf — d. Very active on most days  
 
10. How thick is the roll of fat when you pinch the side of your wai...

SEMANTIC RESULTS:
  1. Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  2. Trans fatt acids - A risk factor for cardiovascular disease.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  3. Trans fatt acids - A risk factor for cardiovascular disease.pdf — recommendation that the contents 

## Step 17: Load semantic re-ranker

Load **CrossEncoder (ms-marco-MiniLM-L-6-v2)** which fine-tunes semantic relevance by directly scoring query-document pairs (not symmetric like embeddings).


In [13]:
from sentence_transformers import CrossEncoder

print("Loading CrossEncoder re-ranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Re-ranker loaded")

def rerank(query, docs, top_k=3):
    # Create pairs of (query, document text)
    pairs = [[query, doc.page_content] for doc in docs]

    # Score each pair
    scores = reranker.predict(pairs)

    # Sort by score highest first
    scored_docs = list(zip(scores, docs))
    scored_docs.sort(key=lambda x: x[0], reverse=True)

    return [doc for score, doc in scored_docs[:top_k]]

# Test it on previous results
reranked_results = rerank(query, results)

print(f"Query: {query}\n")
print("BEFORE reranking:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.metadata['source']} — {doc.page_content[:100]}...")

print("\nAFTER reranking:")
for i, doc in enumerate(reranked_results, 1):
    print(f"  {i}. {doc.metadata['source']} — {doc.page_content[:100]}...")

Loading CrossEncoder re-ranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2107.73it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Re-ranker loaded
Query: I eat biryani and karahi 4 times a week, is that bad for my heart

BEFORE reranking:
  1. Trans fatt acids - A risk factor for cardiovascular disease.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  2. Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf — recommendation that the contents of TFA in human dietary fat should be reduced to less than 4%. Ther...
  3. Caring For Your Heart.pdf — d. Smaller than the size of my thumb  
 
11. How is excess fat distributed across your body? 
 
a. I...
  4. What-Are-the-Warning-Signs-of-Heart-Attack.pdf — journeys with heart disease and stroke 
by joining our Support Network at 
heart.org/SupportNetwork....
  5. Caring For Your Heart.pdf — d. Very active on most days  
 
10. How thick is the roll of fat when you pinch the side of your wai...

AFTER reranking:
  1. Trans fatt acids - A risk factor for cardiovascular disease.pdf — fish sandwi

## Step 18: Test reranking pipeline

Retrieve 6 documents via hybrid search, then re-rank to top 5 using CrossEncoder. Compare before/after rankings.


In [14]:
# Get 6 results from hybrid search instead of 3
results_6 = hybrid_search(query, k=5)

# Rerank down to top 3 using CrossEncoder
reranked_results = rerank(query, results_6, top_k=5)

print(f"Query: {query}\n")
print("TOP 6 from Hybrid Search:")
for i, doc in enumerate(results_6, 1):
    print(f"  {i}. {doc.metadata['source']}")
    print(f"     {doc.page_content[:150]}...")
    print()

print("TOP 3 AFTER CrossEncoder Reranking:")
for i, doc in enumerate(reranked_results, 1):
    print(f"  {i}. {doc.metadata['source']}")
    print(f"     {doc.page_content[:150]}...")
    print()

BM25 RESULTS:
  1. Caring For Your Heart.pdf — d. Smaller than the size of my thumb  
 
11. How is excess fat distributed across your body? 
 
a. I...
  2. What-Are-the-Warning-Signs-of-Heart-Attack.pdf — journeys with heart disease and stroke 
by joining our Support Network at 
heart.org/SupportNetwork....
  3. Caring For Your Heart.pdf — d. Very active on most days  
 
10. How thick is the roll of fat when you pinch the side of your wai...

SEMANTIC RESULTS:
  1. Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  2. Trans fatt acids - A risk factor for cardiovascular disease.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  3. Trans fatt acids - A risk factor for cardiovascular disease.pdf — recommendation that the contents of TFA in human dietary fat should be reduced to less than 4%. Ther...
  4

## Step 19: Generate LLM response

Use **Groq's Llama 3.1 8B** to generate a friendly, conversational answer grounded in retrieved documents. Temperature=0.7 for balanced creativity and consistency.


In [15]:
from langchain_groq import ChatGroq
from rich.console import Console
from rich.markdown import Markdown

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.7
)

# Build context from reranked results
context = "\n\n---\n\n".join([
    f"Source: {doc.metadata['source']}\n{doc.page_content}"
    for doc in reranked_results
])

prompt = f"""You are a friendly heart health assistant. Answer the patient's question using the guidelines below.

GUIDELINES:
{context}

PATIENT: "{query}"

RULES:
- Base your answer ONLY on facts in the guidelines above
- Use warm, friendly, conversational language
- Keep it simple — like talking to a friend
- Do NOT add medical advice not found in the guidelines

FORMAT:

**Assessment:**
[Write 1-2 friendly sentences summarizing what the guidelines say about their situation]

**Recommendations:**
- [List recommendations only if they appear in guidelines]
- [If no recommendations, say "The guidelines don't provide specific recommendations for this"]

**Disclaimer:** This is based on medical guidelines. Please consult a real doctor for personalized advice.

ANSWER:"""

response = llm.invoke(prompt)
console = Console()
md = Markdown(response.content)
console.print(md)


Assessment: It seems like you enjoy eating biryani and karahi quite frequently, which might be a concern for your  
heart health. The guidelines suggest that high consumption of certain foods, such as vanaspati ghee, which is      
commonly used in cooking, can increase the risk of cardiovascular disease (CVD).                                   

Recommendations: The guidelines recommend a healthy approach to fish consumption, suggesting that increased intake 
of tuna and non-fried fish with presumably small amounts of TFA (Trans Fatty Acids) would be a good choice.        
However, there are no specific recommendations in the guidelines regarding biryani and karahi.                     

Disclaimer: This is based on medical guidelines. Please consult a real doctor for personalized advice.

## Step 20: Evaluate Faithfulness

Measure how factually accurate the LLM response is relative to source documents:
1. Extract claims from the answer
2. Verify each claim against context using **Groq's Llama 3.3 70B**
3. Calculate: (supported claims / total claims) × 100%

**Metric**: Faithfulness %


In [16]:
def evaluate_faithfulness(query, context, answer):
    
    # Create FRESH instance every call — prevents context leak
    fresh_judge = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=GROQ_API_KEY,
        temperature=0
    )

    # Step 1 - Extract claims from the answer
    claims_prompt = f"""Extract all factual claims from this answer as a numbered list.
Each claim should be a single, verifiable statement.

Answer: {answer}

Return ONLY a numbered list of claims, nothing else."""

    claims_response = fresh_judge.invoke(claims_prompt)
    claims_text = claims_response.content
    print("EXTRACTED CLAIMS:")
    print(claims_text)
    print()

    # Step 2 - Verify each claim against context
    verify_prompt = f"""You are a fact checker. Given the following context from medical guidelines,
verify each claim and return a JSON with this exact format:
{{"claims": [{{"claim": "claim text", "supported": true/false, "reason": "why"}}]}}

Context:
{context}

Claims to verify:
{claims_text}

Return ONLY the JSON, no other text."""

    verify_response = fresh_judge.invoke(verify_prompt)

    import json
    import re

    try:
        json_text = re.search(r'\{.*\}', verify_response.content, re.DOTALL).group()
        verification = json.loads(json_text)
        claims = verification['claims']
        supported = sum(1 for c in claims if c['supported'])
        total = len(claims)
        score = (supported / total) * 100 if total > 0 else 0

        print(f"FAITHFULNESS SCORE: {supported}/{total} claims supported = {score:.1f}%")
        return score, claims
    except:
        print("Could not parse verification response")
        print(verify_response.content)
        return 0, []

# Test it
faithfulness_score, claims = evaluate_faithfulness(query, context, response.content)

EXTRACTED CLAIMS:
1. High consumption of vanaspati ghee can increase the risk of cardiovascular disease (CVD).
2. Vanaspati ghee is commonly used in cooking.
3. The guidelines recommend a healthy approach to fish consumption.
4. Increased intake of tuna and non-fried fish with small amounts of TFA (Trans Fatty Acids) is a good choice.
5. There are no specific recommendations in the guidelines regarding biryani and karahi.
6. Medical guidelines suggest that certain foods can increase the risk of cardiovascular disease (CVD).

FAITHFULNESS SCORE: 6/6 claims supported = 100.0%


## Step 21: Evaluate Relevancy

Measure how well the answer addresses the original query:
1. Generate 3 new questions from the answer
2. Compute cosine similarity between each generated question and original query
3. Average similarity scores and convert to percentage

**Metric**: Relevancy %


In [17]:
def evaluate_relevancy(query, answer):

    # Step 1 - Generate 3 questions from the answer
    questions_prompt = f"""Based on this answer, generate 3 different questions that this answer addresses.
Return ONLY a numbered list of 3 questions, nothing else.

Answer: {answer}"""

    questions_response = llm.invoke(questions_prompt)
    generated_questions = questions_response.content.strip().split('\n')
    generated_questions = [q for q in generated_questions if q.strip()][:3]

    print("GENERATED QUESTIONS:")
    for q in generated_questions:
        print(q)
    print()

    # Step 2 - Compute cosine similarity between generated questions and original query
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    original_vector = embedding_model.encode([query])
    question_vectors = embedding_model.encode(generated_questions)

    similarities = cosine_similarity(original_vector, question_vectors)[0]
    avg_relevancy = np.mean(similarities) * 100

    print(f"Similarity scores: {[f'{s:.3f}' for s in similarities]}")
    print(f"RELEVANCY SCORE: {avg_relevancy:.1f}%")

    return avg_relevancy

# Test it
relevancy_score = evaluate_relevancy(query, response.content)

GENERATED QUESTIONS:
1. What potential health risks are associated with frequently eating biryani and karahi?
2. According to medical guidelines, what type of fish consumption is recommended for heart health?
3. Can the guidelines provide specific advice on reducing the risks associated with eating biryani and karahi?

Similarity scores: ['0.822', '0.629', '0.765']
RELEVANCY SCORE: 73.9%


## Step 22: Save BM25 index for deployment

Serialize the BM25 retriever to `bm25_data.pkl` for use in Flask backend. This avoids the need to rebuild BM25 on application startup.


In [ ]:
import pickle

bm25_pkl_path = r"D:\Classess\ITA\Rag_Project\Heart Attack Project\bm25_data.pkl"

# Save bm25 data first
with open(bm25_pkl_path, "wb") as f:
    pickle.dump({"docs": docs_for_bm25}, f)
print("✅ bm25_data.pkl saved")



✅ bm25_data.pkl saved


## Step 23: Test single query on Pinecone

Test the end-to-end pipeline with a single test query to verify Pinecone semantic search is working correctly.


In [ ]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer


pc = Pinecone(PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

query = "I smoke 5 cigarettes a day. What is my risk of heart disease?"
query_vector = model.encode(query).tolist()

results = index.query(vector=query_vector, top_k=5, include_metadata=True)

print("="*80)
print(f"QUERY: {query}")
print("="*80)

for i, match in enumerate(results['matches'], 1):
    print(f"\n--- RESULT {i} ---")
    print(f"Score: {match['score']:.4f}")
    print(f"Source: {match['metadata']['source']}")
    print(f"Page: {match['metadata']['page']}")
    print(f"Content:\n{match['metadata']['text']}")
    print("-"*80)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3282.04it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: I smoke 5 cigarettes a day. What is my risk of heart disease?

--- RESULT 1 ---
Score: 0.7577
Source: Biryani & Diet MVP.pdf
Page: 37
Content:
consumption of as few as one to four cigarettes daily increases the risk of CAD. Smoking acts synergistically 
with oral contraceptive agents, placing younger women at an even higher relative risk. In addition to 
myocardial infarction, cigarette consumption directly relates to increased rates of sudden death, aortic 
aneurysm formation, symptomatic peripheral vascular disease and ischemic stroke. As for coronary artery 
disease, the risk of stroke is directly increased with the number of cigarettes consumed.
23
Pakistan Dietary Guidelines for Better Nutrition
--------------------------------------------------------------------------------

--- RESULT 2 ---
Score: 0.7486
Source: 3rd-Hypertension-Guideline-2018-PHL.pdf
Page: 40
Content:
Cessation of cigarette smoking constitutes the single most important preventive measure for 
CAD. Person

## Step 24: Evaluate 10 diverse test queries

Run the full pipeline (hybrid search → rerank → LLM → evaluate) on 10 diverse queries covering:
- Pakistani foods
- Hypertension  
- Heart attack symptoms
- Risk factors & lifestyle
- Prevention strategies

This provides quantitative baseline metrics (Faithfulness %, Relevancy %) for the current system.


In [19]:
test_queries = [
    "I smoke 10 cigarettes a day and eat fried food, am I at risk of heart attack?",
    "My father had a heart attack at 55, am I at risk?",
    "I have frequent headaches and dizziness, could it be high blood pressure?",
    "I don't exercise at all and sit 10 hours a day, what's my risk?",
    "I eat biryani and karahi 4 times a week, is that bad for my heart?",
    "What are the warning signs of a heart attack?",
    "How much exercise do I need to protect my heart?",
    "What foods should I avoid to prevent heart disease?",
    "I am obese and have diabetes, what is my heart risk?",
    "How does stress affect heart health?"
]

results_table = []

for q in test_queries:
    print(f"\nEvaluating: {q[:60]}...")

    # Retrieve and rerank
    hybrid_results_q = hybrid_search(q, k=6)
    reranked_q = rerank(q, hybrid_results_q, top_k=3)

    # Build context
    context_q = "\n\n---\n\n".join([
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in reranked_q
    ])

    # Generate answer
    prompt = f"""You are a friendly heart health assistant. Answer the patient's question using the guidelines below.

GUIDELINES:
{context_q}

PATIENT: "{q}"

RULES:
- Base your answer ONLY on facts in the guidelines above
- Use warm, friendly, conversational language
- Keep it simple — like talking to a friend
- Do NOT add medical advice not found in the guidelines

FORMAT:

**Assessment:**
[Write 1-2 friendly sentences summarizing what the guidelines say about their situation]

**Recommendations:**
- [List recommendations only if they appear in guidelines]
- [If no recommendations, say "The guidelines don't provide specific recommendations for this"]

**Disclaimer:** This is based on medical guidelines. Please consult a real doctor for personalized advice.

ANSWER:"""
    response_q = llm.invoke(prompt)
    answer_q = response_q.content

    # ADD THIS
    print(f"\n--- LLM Response ---\n{answer_q}\n--------------------")
    
    # Evaluate
    f_score, _ = evaluate_faithfulness(q, context_q, answer_q)
    r_score = evaluate_relevancy(q, answer_q)

    results_table.append({
        "query": q[:60],
        "faithfulness": round(f_score, 1),
        "relevancy": round(r_score, 1)
    })
    print(f"✅ F: {f_score:.1f}% | R: {r_score:.1f}%")

# Print final table
print("\n" + "="*70)
print("EVALUATION RESULTS TABLE")
print("="*70)
print(f"{'Query':<45} {'Faithfulness':>12} {'Relevancy':>10}")
print("-"*70)
for r in results_table:
    print(f"{r['query']:<45} {r['faithfulness']:>11}% {r['relevancy']:>9}%")

import numpy as np
avg_f = np.mean([r['faithfulness'] for r in results_table])
avg_r = np.mean([r['relevancy'] for r in results_table])
print("-"*70)
print(f"{'AVERAGE':<45} {avg_f:>11.1f}% {avg_r:>9.1f}%")


Evaluating: I smoke 10 cigarettes a day and eat fried food, am I at risk...
BM25 RESULTS:
  1. Caring For Your Heart.pdf — c. I do not have much excess fat  
 
12. What kind of stress do you experience, and how often?  
 
a...
  2. Caring For Your Heart.pdf — b. I am a pre-menopausal woman, with diabetes  
 
c. I am a post-menopausal woman  
 
d. None of the...
  3. What-Are-the-Warning-Signs-of-Heart-Attack.pdf — as shortness of breath, nausea/vomiting and 
back or jaw pain.
What Are the 
Warning Signs of 
Heart...

SEMANTIC RESULTS:
  1. Trans fatt acids - A risk factor for cardiovascular disease.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  2. Trans fatt acids - A risk factor for cardiovascular disease by AKU.pdf — fish sandwiches was associated with increased 
risk of CVD.27  Therefore, increased intake of 
tuna ...
  3. 3rd-Hypertension-Guideline-2018-PHL.pdf — Cessation of cigarette smoking constitutes the single 

## Step 25: Chunk Size Ablation Study (Part 1)

Test how **different chunk sizes** affect system performance:
- **1000 tokens / 200 overlap** (current)
- **500 tokens / 100 overlap** (recommended)
- **300 tokens / 50 overlap** (small - potentially more precise)

Run evaluation on 20 queries per strategy. Compare Faithfulness and Relevancy metrics.


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import re
import time
import json

chunking_strategies = [
    {"name": "Current (1000/200)",    "chunk_size": 1000, "overlap": 200},
    {"name": "Recommended (500/100)", "chunk_size": 500,  "overlap": 100},
    {"name": "Small (300/50)",        "chunk_size": 300,  "overlap": 50},
]

ablation_queries = [
    # Pakistani Dishes (5)
    "I eat mutton karahi 3 times a week. Is that bad for my heart?",
    "Which is healthier - chicken biryani or chicken karahi?",
    "I eat chapli kabab every day. Will this give me a heart attack?",
    "Is beef nihari safe for someone with high cholesterol?",
    "I eat haleem for dinner every night. Is that okay for weight loss?",
    
    # Hypertension (3)
    "My home BP is 135/85 on average for 7 days. Do I have hypertension?",
    "What is the normal blood pressure for a 65 year old man?",
    "Can high blood pressure cause headaches and dizziness?",
    
    # Heart Attack & Symptoms (3)
    "What are the warning signs of a heart attack in women?",
    "I have shortness of breath but no chest pain. Should I worry?",
    "Can a heart attack happen without any symptoms?",
    
    # Risk Factors (3)
    "I smoke 5 cigarettes a day. How much does this increase my heart attack risk?",
    "My father died of a heart attack at 52. Am I at high risk?",
    "Is vanaspati ghee worse than desi ghee for my heart?",
    
    # Treatment & Medications (2)
    "I am pregnant and have high BP. Which blood pressure medicines are safe?",
    "Can a nurse or pharmacist prescribe my blood pressure medication?",
    
    # Lifestyle & Prevention (2)
    "How much exercise do I need per week to protect my heart?",
    "I sit at a desk for 10 hours every day. How bad is this for my heart?",
    
    # General (2)
    "How does stress affect heart health?",
    "What foods should I avoid to prevent heart disease?",
]

ablation_results = []

judge_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0
)

def evaluate_faithfulness_ablation(query, context, answer):
    claims_prompt = f"""Extract all factual claims from this answer as a numbered list.
Each claim should be a single, verifiable statement.

Answer: {answer}

Return ONLY a numbered list of claims, nothing else."""

    claims_text = judge_llm.invoke(claims_prompt).content
    
    if not claims_text.strip():
        print("Could not extract claims")
        return 0.0

    verify_prompt = f"""You are a fact checker. Given the following context from medical guidelines,
verify each claim and return a JSON with this exact format.
Use ONLY true or false for supported field.

Context:
{context[:2000]}

Claims to verify:
{claims_text}

Return ONLY the JSON. Format:
{{"claims": [{{"claim": "claim text", "supported": true/false}}]}}"""

    verify_text = judge_llm.invoke(verify_prompt).content

    try:
        json_text = re.search(r'\{.*\}', verify_text, re.DOTALL).group()
        json_text = json_text.replace('"supported": not verified', '"supported": false')
        json_text = json_text.replace('"supported": Not in context', '"supported": false')
        verification = json.loads(json_text)
        claims = verification['claims']
        supported = sum(1 for c in claims if c.get('supported', False))
        total = len(claims)
        score = (supported / total) * 100 if total > 0 else 0
        return round(score, 1)
    except Exception as e:
        print(f"Failed to parse evaluation: {e}")
        return 0.0

for strategy in chunking_strategies:
    print(f"\n{'='*60}")
    print(f"Testing: {strategy['name']}")
    print('='*60)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=strategy['chunk_size'],
        chunk_overlap=strategy['overlap'],
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    test_chunks = []
    test_metadata = []
    for i in range(len(chunks)):
        split = splitter.split_text(chunks[i])
        for s in split:
            if s.strip():
                test_chunks.append(s)
                test_metadata.append(chunks_metadata[i])

    print(f"Total chunks: {len(test_chunks)}")

    test_docs = [
        Document(page_content=test_chunks[i], metadata=test_metadata[i])
        for i in range(len(test_chunks))
    ]
    test_bm25 = BM25Retriever.from_documents(test_docs, k=5)

    f_scores = []
    r_scores = []

    for q in ablation_queries:
        print(f"\n{'─'*60}")
        print(f"QUERY: {q}")
        print('─'*60)

        bm25_res = test_bm25.invoke(q)
        sem_res = semantic_search(q, k=5)

        scores = {}
        all_docs = {}
        for rank, doc in enumerate(bm25_res):
            doc_id = doc.page_content[:100]
            scores[doc_id] = scores.get(doc_id, 0) + 1/(rank + 60)
            all_docs[doc_id] = doc
        for rank, doc in enumerate(sem_res):
            doc_id = doc.page_content[:100]
            scores[doc_id] = scores.get(doc_id, 0) + 1/(rank + 60)
            all_docs[doc_id] = doc
        sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        hybrid_res = [all_docs[doc_id] for doc_id, _ in sorted_docs[:6]]

        final_docs = rerank(q, hybrid_res, top_k=3)

        ctx = "\n\n---\n\n".join([
            f"Source: {doc.metadata['source']}\n{doc.page_content}"
            for doc in final_docs
        ])

        prompt = f"""You are a friendly heart health assistant. Answer the patient's question using the guidelines below.

GUIDELINES:
{ctx}

PATIENT: "{q}"

RULES:
- Base your answer ONLY on facts in the guidelines above
- Use warm, friendly, conversational language
- Keep it simple — like talking to a friend
- Do NOT add medical advice not found in the guidelines

FORMAT:

**Assessment:**
[Write 1-2 friendly sentences summarizing what the guidelines say about their situation]

**Recommendations:**
- [List recommendations only if they appear in guidelines]
- [If no recommendations, say "The guidelines don't provide specific recommendations for this"]

**Disclaimer:** This is based on medical guidelines. Please consult a real doctor for personalized advice.

ANSWER:"""

        answer = llm.invoke(prompt).content

        # Print the LLM answer to terminal
        print(f"\nLLM ANSWER:")
        print(answer)
        print()

        f_score = evaluate_faithfulness_ablation(q, ctx, answer)
        r_score = evaluate_relevancy(q, answer)

        f_scores.append(f_score)
        r_scores.append(r_score)
        print(f"✅ F: {f_score:.1f}% | R: {r_score:.1f}%")

    avg_f = np.mean(f_scores)
    avg_r = np.mean(r_scores)

    ablation_results.append({
        "strategy": strategy['name'],
        "chunks": len(test_chunks),
        "avg_faithfulness": round(avg_f, 1),
        "avg_relevancy": round(avg_r, 1)
    })
    print(f"\n{'='*60}")
    print(f"STRATEGY AVERAGE — F: {avg_f:.1f}% | R: {avg_r:.1f}%")

# Final table
print(f"\n{'='*70}")
print("CHUNK SIZE ABLATION STUDY")
print('='*70)
print(f"{'Strategy':<25} {'Chunks':>8} {'Faithfulness':>14} {'Relevancy':>10}")
print('-'*70)
for r in ablation_results:
    print(f"{r['strategy']:<25} {r['chunks']:>8} {r['avg_faithfulness']:>13}% {r['avg_relevancy']:>9}%")


Testing: Current (1000/200)
Total chunks: 4413

────────────────────────────────────────────────────────────
QUERY: I eat mutton karahi 3 times a week. Is that bad for my heart?
────────────────────────────────────────────────────────────

LLM ANSWER:
**Assessment:**
According to the guidelines, Mutton Karahi has a high fat content of 20.35%, which is above the recommended limit. This means you should limit eating Mutton Karahi to occasional eating.

**Recommendations:**
- Limit Mutton Karahi to occasional eating.
 
**Disclaimer:** This is based on medical guidelines. Please consult a real doctor for personalized advice.

GENERATED QUESTIONS:
1. What is the recommended limit for fat content in a dish like Mutton Karahi?
2. How often should someone be eating Mutton Karahi based on its high fat content?
3. What guidelines are being referred to in the assessment of Mutton Karahi's nutritional content?

Similarity scores: ['0.752', '0.836', '0.724']
RELEVANCY SCORE: 77.1%
✅ F: 75.0% | R: 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knc0xnr4f2jr062sc4v66k8b` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99792, Requested 774. Please try again in 8m9.024s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Step 27: Build V1 - Fixed chunking + BM25 only

**Baseline retrieval (no Pinecone)**:
- Use simple character-based splitting (fixed 300-token chunks)
- Only BM25 keyword search (no semantic embeddings)
- This is the cheapest, simplest baseline for comparison

Used for 4-way ablation study.


In [45]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

print("Building V1 - Fixed chunks...")
fixed_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=300,
    chunk_overlap=50
)

fixed_chunks = []
fixed_metadata = []

for i, text in enumerate(all_texts):
    split_texts = fixed_splitter.split_text(text)
    for chunk in split_texts:
        if chunk.strip():
            fixed_chunks.append(chunk)
            fixed_metadata.append(doc_metadata[i])

print(f"Fixed chunks created: {len(fixed_chunks)}")

fixed_docs = [
    Document(page_content=fixed_chunks[i], metadata=fixed_metadata[i])
    for i in range(len(fixed_chunks))
]

fixed_bm25 = BM25Retriever.from_documents(fixed_docs, k=5)

# V1 = Fixed BM25 only, no Pinecone
def v1_search(query, k=3):
    return fixed_bm25.invoke(query)[:k]

print("✅ V1 ready")

Created a chunk of size 394, which is longer than the specified 300
Created a chunk of size 331, which is longer than the specified 300
Created a chunk of size 819, which is longer than the specified 300


Building V1 - Fixed chunks...
Fixed chunks created: 440
✅ V1 ready


## Step 28: Ablation Study - Compare 4 Retrieval Versions

**4-way comparison** on same 10 test queries:
- **V1**: Fixed chunks + BM25 only
- **V2**: Recursive chunks + Semantic (Pinecone) only
- **V3**: Recursive + Hybrid (RRF) - NO reranking
- **V4**: Recursive + Hybrid + CrossEncoder reranking 

Expected winner: V4 (combines all techniques)

Shows impact of:
- Semantic vs keyword search
- Hybrid fusion
- Semantic re-ranking



In [ ]:
ablation_queries = [
    "I smoke 10 cigarettes a day and eat fried food, am I at risk of heart attack?",
    "My father had a heart attack at 55, am I at risk?",
    "I have frequent headaches and dizziness, could it be high blood pressure?",
    "I don't exercise at all and sit 10 hours a day, what's my risk?",
    "I eat biryani and karahi 4 times a week, is that bad for my heart?",
    "What are the warning signs of a heart attack?",
    "How much exercise do I need to protect my heart?",
    "What foods should I avoid to prevent heart disease?",
    "I am obese and have diabetes, what is my heart risk?",
    "How does stress affect heart health?"
]

def hybrid_search_with_rerank(query, k=3):
    candidates = hybrid_search(query, k=10)
    rerank_pairs = [(query, doc.page_content) for doc in candidates]
    rerank_scores = reranker.predict(rerank_pairs)
    reranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)[:k]
    return [doc for doc, score in reranked]

def run_ablation_version(version_name, search_fn, queries):
    print(f"\n{'='*60}")
    print(f"TESTING: {version_name}")
    print('='*60)

    f_scores = []
    r_scores = []

    for q in queries:
        print(f"\nQuery: {q[:50]}...")

        docs = search_fn(q)

        ctx = "\n\n---\n\n".join([
            f"Source: {doc.metadata['source']}\n{doc.page_content}"
            for doc in docs
        ])

        prompt = f"""You are a friendly heart health assistant. Answer the patient's question using the guidelines below.

GUIDELINES:
{ctx}

PATIENT: "{q}"

RULES:
- Base your answer ONLY on facts in the guidelines above
- Use warm, friendly, conversational language
- Keep it simple — like talking to a friend
- Do NOT add medical advice not found in the guidelines

FORMAT:

**Assessment:**
[Write 1-2 friendly sentences summarizing what the guidelines say about their situation]

**Recommendations:**
- [List recommendations only if they appear in guidelines]
- [If no recommendations, say "The guidelines don't provide specific recommendations for this"]

**Disclaimer:** This is based on medical guidelines. Please consult a real doctor for personalized advice.

ANSWER:"""

        ans = llm.invoke(prompt).content

        # Safely handle both tuple and float return from evaluate_faithfulness
        faith_result = evaluate_faithfulness(q, ctx, ans)
        f_score = faith_result[0] if isinstance(faith_result, tuple) else faith_result

        r_score = evaluate_relevancy(q, ans)

        f_scores.append(f_score)
        r_scores.append(r_score)
        print(f"F: {f_score:.1f}% | R: {r_score:.1f}%")

    avg_f = sum(f_scores) / len(f_scores)
    avg_r = sum(r_scores) / len(r_scores)
    print(f"\nAVERAGE — F: {avg_f:.1f}% | R: {avg_r:.1f}%")
    return avg_f, avg_r

# Run all 4 versions
v1_f, v1_r = run_ablation_version(
    "V1: Fixed Chunking + BM25 Only",
    lambda q: v1_search(q, k=3),
    ablation_queries
)

v2_f, v2_r = run_ablation_version(
    "V2: Recursive Chunking + Semantic Only",
    lambda q: semantic_search(q, k=3),
    ablation_queries
)

v3_f, v3_r = run_ablation_version(
    "V3: Recursive + Hybrid (BM25 + Semantic)",
    lambda q: hybrid_search(q, k=3),
    ablation_queries
)

v4_f, v4_r = run_ablation_version(
    "V4: Recursive + Hybrid + Re-ranking",
    lambda q: hybrid_search_with_rerank(q, k=3),
    ablation_queries
)

# Final table
print(f"\n{'='*70}")
print("ABLATION STUDY RESULTS")
print('='*70)
print(f"{'Version':<40} {'Faithfulness':>14} {'Relevancy':>10}")
print('-'*70)
print(f"{'V1: Fixed + BM25 Only':<40} {v1_f:>13.1f}% {v1_r:>9.1f}%")
print(f"{'V2: Recursive + Semantic Only':<40} {v2_f:>13.1f}% {v2_r:>9.1f}%")
print(f"{'V3: Recursive + Hybrid':<40} {v3_f:>13.1f}% {v3_r:>9.1f}%")
print(f"{'V4: Recursive + Hybrid + Rerank':<40} {v4_f:>13.1f}% {v4_r:>9.1f}%")